# N03: SigLIP, searching the image itself

A different branch: no text about the painting at all. SigLIP was trained on image-caption pairs so that a sentence
and a matching image land close together; we compare the visitor's query (text tower) with each painting (image
tower). R4 is compared with the baseline R1, not with R3 (it changes the encoder, the modality and the text at once).

The image embeddings are computed once and cached; at query time only the query goes through the text tower.

In [ ]:

from deep_learning_project_waerniers_lamy.nlp import utils as u

In [ ]:
paintings = u.load_corpus("dev")
descriptions = u.load_descriptions()
queries = u.QUERIES["text"].tolist()
scores = {
    "R0 random": u.rung_random(paintings, queries),
    "R1 metadata": u.rung_metadata(paintings, queries),
    "R2 description": u.rung_description(paintings, queries, descriptions),
    "R3 description+metadata": u.rung_description_metadata(paintings, queries, descriptions),
    "R4 siglip image": u.rung_siglip(paintings, queries),
}
u.judge_with_llm("dev", scores, paintings)

### Judging

Every painting in the new top 5 of a rung is graded by an LLM judge (`u.JUDGE_MODEL`, another model family than the
description generator). It sees **only the query and the image**, answers two yes/no questions (is the main subject
there? does it match the whole query?) and the grade follows by rule: 0, 1 or 2. Grades are saved in
`results/nlp/qrels_dev.json`; already graded pairs are never sent again. The judge is validated against 150 of your own
grades in N04 before any result is frozen.

## Results

Pre-registered comparison 2 of 3: **does a visual embedding beat metadata?** (R4 vs R1).

In [ ]:
per_query = u.per_query_scores("dev", scores, paintings)
table = u.summary_table(per_query)
u.plot_ladder(table, "dev: + SigLIP")
display(table)
u.compare(per_query, [("R4 siglip image", "R1 metadata")])

In [ ]:
per_query.groupby(["rung", "category"])["ndcg@5"].mean().unstack().round(3)